In [34]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [15]:
%%html
<style>
div.jp-OutputArea-output pre,
div.output_area pre {
    white-space: pre !important;
    overflow-x: auto;
}
</style>

In [11]:
spark = (SparkSession
    .builder
    .master("local[*]")
    .appName("PPyC")
    .getOrCreate()
)

In [10]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("/home/jovyan/work/data/supermercados")
DATA_DIR.mkdir(parents=True, exist_ok=True)

url = "https://raw.githubusercontent.com/ianmazzola/SupermercadosArgentina/refs/heads/main/ventas-totales-supermercados-2%20(1).csv"
csv_path = DATA_DIR / "ventas_supermercados.csv"

if not csv_path.exists():
    urlretrieve(url, csv_path)

csv_path

PosixPath('/home/jovyan/work/data/supermercados/ventas_supermercados.csv')

In [28]:
df = (spark
    .read
    .option("header", "true")
    .option("sep", ",")
    .option("inferSchema", "true")
    .csv("/home/jovyan/work/data/supermercados/ventas_supermercados.csv")
)

In [29]:
df.show(5, False)

+-------------+-------------------------+-------------------------+--------------------------+--------------------+---------------+-------------------------+--------------------+-----------------+-----------------+------------------+------------------------------+---------------------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------------------+-----------------------------+-----------------------------------+----------------------------+------------------+
|indice_tiempo|ventas_precios_corrientes|ventas_precios_constantes|ventas_totales_canal_venta|salon_ventas        |canales_on_line|ventas_totales_medio_pago|efectivo            |tarjetas_debito  |tarjetas_credito |otros_medios      |ventas_totales_grupo_articulos|subtotal_ventas_alimentos_bebidas|bebidas           |almacen           |panaderia         |lacteos           |carnes            |verduleria_fruteria|alimentos_preparados_rotiser

In [30]:
df.printSchema()

root
 |-- indice_tiempo: date (nullable = true)
 |-- ventas_precios_corrientes: double (nullable = true)
 |-- ventas_precios_constantes: double (nullable = true)
 |-- ventas_totales_canal_venta: double (nullable = true)
 |-- salon_ventas: double (nullable = true)
 |-- canales_on_line: double (nullable = true)
 |-- ventas_totales_medio_pago: double (nullable = true)
 |-- efectivo: double (nullable = true)
 |-- tarjetas_debito: double (nullable = true)
 |-- tarjetas_credito: double (nullable = true)
 |-- otros_medios: double (nullable = true)
 |-- ventas_totales_grupo_articulos: double (nullable = true)
 |-- subtotal_ventas_alimentos_bebidas: double (nullable = true)
 |-- bebidas: double (nullable = true)
 |-- almacen: double (nullable = true)
 |-- panaderia: double (nullable = true)
 |-- lacteos: double (nullable = true)
 |-- carnes: double (nullable = true)
 |-- verduleria_fruteria: double (nullable = true)
 |-- alimentos_preparados_rotiseria: double (nullable = true)
 |-- articulos_li

In [31]:
df.schema

StructType([StructField('indice_tiempo', DateType(), True), StructField('ventas_precios_corrientes', DoubleType(), True), StructField('ventas_precios_constantes', DoubleType(), True), StructField('ventas_totales_canal_venta', DoubleType(), True), StructField('salon_ventas', DoubleType(), True), StructField('canales_on_line', DoubleType(), True), StructField('ventas_totales_medio_pago', DoubleType(), True), StructField('efectivo', DoubleType(), True), StructField('tarjetas_debito', DoubleType(), True), StructField('tarjetas_credito', DoubleType(), True), StructField('otros_medios', DoubleType(), True), StructField('ventas_totales_grupo_articulos', DoubleType(), True), StructField('subtotal_ventas_alimentos_bebidas', DoubleType(), True), StructField('bebidas', DoubleType(), True), StructField('almacen', DoubleType(), True), StructField('panaderia', DoubleType(), True), StructField('lacteos', DoubleType(), True), StructField('carnes', DoubleType(), True), StructField('verduleria_fruteria'

In [36]:
# transformaciones
df_final = df.select(
    F.col("indice_tiempo"),
    F.col("efectivo"),
    F.col("otros")
)

In [37]:
# acciones
df_final.show(5, False)

+-------------+--------------------+------------------+
|indice_tiempo|efectivo            |otros             |
+-------------+--------------------+------------------+
|2017-01-01   |1.0230100130314E7   |2806660.3093699994|
|2017-02-01   |9719067.340596005   |2354084.9006299996|
|2017-03-01   |1.0244442508284997E7|2266189.276857    |
|2017-04-01   |1.0312632366428997E7|2146755.8872700003|
|2017-05-01   |1.014761334208E7    |2117971.4826695994|
+-------------+--------------------+------------------+
only showing top 5 rows



In [40]:
df_final.explain()

== Physical Plan ==
FileScan csv [indice_tiempo#1016,efectivo#1023,otros#1039] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/supermercados/ventas_supermercados.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<indice_tiempo:date,efectivo:double,otros:double>




In [41]:
# transformacion
df_final = (df_final
    .where(
        F.year(F.col("indice_tiempo")) > F.lit(2018)
    )
)

In [42]:
df_final.explain()

== Physical Plan ==
*(1) Filter (isnotnull(indice_tiempo#1016) AND (year(indice_tiempo#1016) > 2018))
+- FileScan csv [indice_tiempo#1016,efectivo#1023,otros#1039] Batched: false, DataFilters: [isnotnull(indice_tiempo#1016), (year(indice_tiempo#1016) > 2018)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/supermercados/ventas_supermercados.csv], PartitionFilters: [], PushedFilters: [IsNotNull(indice_tiempo)], ReadSchema: struct<indice_tiempo:date,efectivo:double,otros:double>




In [43]:
# accion
df_final.count()

68

In [44]:
type(df_final)

pyspark.sql.dataframe.DataFrame

In [45]:
df.createTempView("transacciones")

In [50]:
df_final_sql = spark.sql("""
SELECT
    indice_tiempo, efectivo, otros
FROM
    transacciones
WHERE
    year(indice_tiempo) > 2018
""")

In [47]:
df_final_sql.count()

68

In [48]:
df_final_sql.explain()

== Physical Plan ==
*(1) Filter (isnotnull(indice_tiempo#1016) AND (year(indice_tiempo#1016) > 2018))
+- FileScan csv [indice_tiempo#1016,efectivo#1023,otros#1039] Batched: false, DataFilters: [isnotnull(indice_tiempo#1016), (year(indice_tiempo#1016) > 2018)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/supermercados/ventas_supermercados.csv], PartitionFilters: [], PushedFilters: [IsNotNull(indice_tiempo)], ReadSchema: struct<indice_tiempo:date,efectivo:double,otros:double>




In [59]:
fact_medios_pago = spark.sql("""
SELECT A.*
FROM
    (SELECT indice_tiempo, 'efectivo' as tipo_pago, efectivo as monto 
    FROM TRANSACCIONES
    UNION
    SELECT indice_tiempo, 'tarjetas_debito' as tipo_pago, tarjetas_debito as monto
    FROM TRANSACCIONES
    UNION
    SELECT indice_tiempo, 'tarjetas_credito' as tipo_pago, tarjetas_credito as monto
    FROM TRANSACCIONES
    UNION
    SELECT indice_tiempo, 'otros_medios' as tipo_pago, otros_medios as monto
    FROM TRANSACCIONES) A
ORDER  BY A.indice_tiempo desc
""")

In [57]:
fact_medios_pago.show(5)

+-------------+----------------+--------------------+
|indice_tiempo|       tipo_pago|               monto|
+-------------+----------------+--------------------+
|   2024-08-01| tarjetas_debito|4.5599748846533006E8|
|   2024-08-01|tarjetas_credito| 7.700564276983199E8|
|   2024-08-01|    otros_medios|1.5174008300999996E8|
|   2024-08-01|        efectivo| 2.896681181498199E8|
|   2024-07-01| tarjetas_debito|4.6180271050931984E8|
+-------------+----------------+--------------------+
only showing top 5 rows



In [61]:
medios_pago = ["efectivo", "tarjetas_credito", "tarjetas_debito", "otros_medios"]
querys = []
for colname in medios_pago:
    querys.append(f"""
    SELECT
        indice_tiempo, '{colname}' as tipo_pago, {colname} as monto 
    FROM
        TRANSACCIONES
    """)

In [64]:
final_query = "UNION".join(querys)
fact_medios_pago = spark.sql(f"""
SELECT A.*
FROM
    ({final_query}) A
ORDER  BY A.indice_tiempo desc
""")

In [66]:
fact_medios_pago.show(2, 0)

+-------------+------------+--------------------+
|indice_tiempo|tipo_pago   |monto               |
+-------------+------------+--------------------+
|2024-08-01   |efectivo    |2.896681181498199E8 |
|2024-08-01   |otros_medios|1.5174008300999996E8|
+-------------+------------+--------------------+
only showing top 2 rows



In [69]:
transacciones_efectivo = df.select(
    F.col("indice_tiempo"),
    F.lit('efectivo').alias("tipo_pago"),
    F.col("efectivo").alias("monto")
)
transacciones_otros_medios = df.select(
    F.col("indice_tiempo"),
    F.lit('otros_medios').alias("tipo_pago"),
    F.col("otros_medios").alias("monto")
)
transacciones_tarjetas_debito = df.select(
    F.col("indice_tiempo"),
    F.lit('tarjetas_debito').alias("tipo_pago"),
    F.col("tarjetas_debito").alias("monto")
)

transacciones_tarjetas_credito = df.select(
    F.col("indice_tiempo"),
    F.lit('tarjetas_credito').alias("tipo_pago"),
    F.col("tarjetas_credito").alias("monto")
) 

In [70]:
fact_medios_pago = transacciones_efectivo.union(
    transacciones_otros_medios
).union(
    transacciones_tarjetas_debito    
).union(
    transacciones_tarjetas_credito 
).orderBy(F.desc(F.col("indice_tiempo")))

In [71]:
fact_medios_pago.show(5, 0)

+-------------+----------------+--------------------+
|indice_tiempo|tipo_pago       |monto               |
+-------------+----------------+--------------------+
|2024-08-01   |tarjetas_credito|7.700564276983199E8 |
|2024-08-01   |tarjetas_debito |4.5599748846533006E8|
|2024-08-01   |otros_medios    |1.5174008300999996E8|
|2024-08-01   |efectivo        |2.896681181498199E8 |
|2024-07-01   |tarjetas_debito |4.6180271050931984E8|
+-------------+----------------+--------------------+
only showing top 5 rows



In [86]:
medios_pago = ["efectivo", "tarjetas_credito", "tarjetas_debito", "otros_medios"]
transaciones_pagos = []
for colname in medios_pago:
    transaciones_pagos.append(df.select(
        F.col("indice_tiempo"),
        F.lit(colname).alias("tipo_pago"),
        F.col(colname).cast("integer").alias("monto")
    ))

In [87]:
from functools import reduce
fact_medios_pago = reduce(lambda a,b: a.union(b), transaciones_pagos)
fact_medios_pago = fact_medios_pago.orderBy(F.desc(F.col("indice_tiempo")))

In [88]:
fact_medios_pago.show(5, 0)

+-------------+----------------+---------+
|indice_tiempo|tipo_pago       |monto    |
+-------------+----------------+---------+
|2024-08-01   |efectivo        |289668118|
|2024-08-01   |tarjetas_debito |455997488|
|2024-08-01   |tarjetas_credito|770056427|
|2024-08-01   |otros_medios    |151740083|
|2024-07-01   |tarjetas_debito |461802710|
+-------------+----------------+---------+
only showing top 5 rows



In [85]:
(fact_medios_pago
    .write
    .mode("overwrite")
    .parquet("/home/jovyan/work/data/output/fact_medios_pago")
)

In [79]:
df.write.parquet("/home/jovyan/work/data/output/raw")